<a href="https://colab.research.google.com/github/venkata18167/CSA6301---THREAT-INTELLIGENCE-AND-NETWORK-SECURITY/blob/main/22_Building_a_Mini_SIEM_Pipeline_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
 import pandas as pd

print("="*75)
print(" MINI SIEM PIPELINE ")
print(" Collect → Normalize → Correlate → Alert ")
print("="*75)


firewall_logs = pd.DataFrame({
    "Time":["10:00:01","10:00:05","10:00:10","10:00:15"],
    "IP":["192.168.1.10","192.168.1.20","192.168.1.10","10.10.10.5"],
    "Action":["ALLOW","ALLOW","BLOCK","ALLOW"]
})

ssh_logs = pd.DataFrame({
    "Time":["10:00:02","10:00:03","10:00:04","10:00:11","10:00:12"],
    "IP":["192.168.1.10","192.168.1.10","192.168.1.10",
          "192.168.1.20","10.10.10.5"],
    "Status":["Failed","Failed","Failed","Success","Failed"]
})

web_logs = pd.DataFrame({
    "Time":["10:00:06","10:00:07","10:00:08","10:00:09"],
    "IP":["192.168.1.10","192.168.1.10",
          "192.168.1.20","10.10.10.5"],
    "Page":["/login","/login","/home","/admin"]
})

print("\nFirewall Logs\n")
print(firewall_logs)

print("\nSSH Logs\n")
print(ssh_logs)

print("\nWeb Logs\n")
print(web_logs)
fw = firewall_logs.rename(columns={
    "Action":"Event"
})

fw["Source"] = "Firewall"

ssh = ssh_logs.rename(columns={
    "Status":"Event"
})

ssh["Source"] = "SSH"

web = web_logs.rename(columns={
    "Page":"Event"
})

web["Source"] = "Web"

fw = fw[["Time","IP","Source","Event"]]
ssh = ssh[["Time","IP","Source","Event"]]
web = web[["Time","IP","Source","Event"]]

normalized = pd.concat([fw,ssh,web],ignore_index=True)

print("\n")
print("="*75)
print("NORMALIZED LOGS")
print("="*75)
print(normalized)

failed_login = ssh_logs[ssh_logs["Status"]=="Failed"]

failed_count = failed_login.groupby("IP").size().reset_index(name="Failed_Logins")

web_count = web_logs.groupby("IP").size().reset_index(name="Web_Requests")

firewall_count = firewall_logs.groupby("IP").size().reset_index(name="Firewall_Events")

report = failed_count.merge(web_count,on="IP",how="outer")
report = report.merge(firewall_count,on="IP",how="outer")

report.fillna(0,inplace=True)


threat_feed = {
    "10.10.10.5":"Known Malware Server",
    "185.220.101.5":"Botnet",
    "192.168.1.50":"Blacklisted Host"
}

intel=[]

for ip in report["IP"]:

    if ip in threat_feed:
        intel.append(threat_feed[ip])
    else:
        intel.append("Not Found")

report["Threat_Intelligence"]=intel


alerts=[]
risk=[]

for _,row in report.iterrows():

    score=0

    if row["Failed_Logins"]>=3:
        score+=40

    if row["Web_Requests"]>=2:
        score+=30

    if row["Threat_Intelligence"]!="Not Found":
        score+=30

    if score>=70:
        alerts.append("HIGH")
    elif score>=40:
        alerts.append("MEDIUM")
    else:
        alerts.append("LOW")

    risk.append(score)

report["Risk Score"]=risk
report["Alert Level"]=alerts


print("\n")
print("="*75)
print("CORRELATED SIEM REPORT")
print("="*75)

print(report)

print("\n")
print("="*75)
print("SECURITY ALERTS")
print("="*75)

high = report[report["Alert Level"]=="HIGH"]

if len(high)==0:
    print("No High Severity Alerts.")

else:

    for _,row in high.iterrows():

        print("IP Address          :",row["IP"])
        print("Failed Logins       :",row["Failed_Logins"])
        print("Web Requests        :",row["Web_Requests"])
        print("Firewall Events     :",row["Firewall_Events"])
        print("Threat Intelligence :",row["Threat_Intelligence"])
        print("Risk Score          :",row["Risk Score"])
        print("Alert Level         :",row["Alert Level"])
        print("-"*50)


report.to_csv("Mini_SIEM_Report.csv",index=False)

print("\nReport Saved As : Mini_SIEM_Report.csv")

print("\nMini SIEM Pipeline Completed Successfully.")

 MINI SIEM PIPELINE 
 Collect → Normalize → Correlate → Alert 

Firewall Logs

       Time            IP Action
0  10:00:01  192.168.1.10  ALLOW
1  10:00:05  192.168.1.20  ALLOW
2  10:00:10  192.168.1.10  BLOCK
3  10:00:15    10.10.10.5  ALLOW

SSH Logs

       Time            IP   Status
0  10:00:02  192.168.1.10   Failed
1  10:00:03  192.168.1.10   Failed
2  10:00:04  192.168.1.10   Failed
3  10:00:11  192.168.1.20  Success
4  10:00:12    10.10.10.5   Failed

Web Logs

       Time            IP    Page
0  10:00:06  192.168.1.10  /login
1  10:00:07  192.168.1.10  /login
2  10:00:08  192.168.1.20   /home
3  10:00:09    10.10.10.5  /admin


NORMALIZED LOGS
        Time            IP    Source    Event
0   10:00:01  192.168.1.10  Firewall    ALLOW
1   10:00:05  192.168.1.20  Firewall    ALLOW
2   10:00:10  192.168.1.10  Firewall    BLOCK
3   10:00:15    10.10.10.5  Firewall    ALLOW
4   10:00:02  192.168.1.10       SSH   Failed
5   10:00:03  192.168.1.10       SSH   Failed
6   10:00:04  